In [15]:
# ============================================================
# CELL 0 — SETUP
# Run this first every time you open the notebook
# ============================================================

# Install required libraries
!pip install MDAnalysis biopython networkx pandas matplotlib -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import networkx as nx
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from Bio import PDB
import MDAnalysis as mda
from MDAnalysis.analysis import rms
from MDAnalysis.transformations import unwrap, fit_rot_trans
import csv
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded")

# ============================================================
# MASTER CONFIG — CHANGE THESE FOR EACH PROTEIN/RUN
# ============================================================

PROTEIN_NAME = 'Chignolin'        # protein name
RUN_NAME     = 'Run5'             # run name
N_RESIDUES   = 10                 # number of residues
CUTOFF       = 8.0                # RIN distance cutoff in Angstroms

# Input files — upload these to Colab before running
TPR_FILE     = '/content/nvt_prod_5.tpr'
XTC_FILE     = '/content/nvt_prod_5.xtc'
NATIVE_PDB   = '/content/5awl.pdb'

# Drive output folder — all results go here
DRIVE_BASE   = f'/content/drive/MyDrive/RIN_Research/{PROTEIN_NAME}/{RUN_NAME}'

# Subfolders — created automatically
FOLDERS = {
    'protein_traj' : f'{DRIVE_BASE}/1_protein_trajectory',
    'frames'       : f'{DRIVE_BASE}/2_pdb_frames',
    'matrices'     : f'{DRIVE_BASE}/3_adjacency_matrices',
    'centrality'   : f'{DRIVE_BASE}/4_centrality',
    'order_params' : f'{DRIVE_BASE}/5_order_parameters',
    'plots'        : f'{DRIVE_BASE}/6_plots',
    'summary'      : f'{DRIVE_BASE}/7_summary',
    'validation'   : f'{DRIVE_BASE}/8_validation',
}

# Create all folders
for folder in FOLDERS.values():
    os.makedirs(folder, exist_ok=True)

print(f"✅ Output folder structure created at:")
print(f"   {DRIVE_BASE}")
for name, path in FOLDERS.items():
    print(f"   └── {name}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ All libraries loaded
✅ Output folder structure created at:
   /content/drive/MyDrive/RIN_Research/Chignolin/Run5
   └── protein_traj
   └── frames
   └── matrices
   └── centrality
   └── order_params
   └── plots
   └── summary
   └── validation


In [16]:
# ============================================================
# CELL 1 — STRIP WATER
# Input:  TPR + XTC files uploaded to Colab
# Output: protein_only.pdb + protein_only.xtc → Drive
# ============================================================

print("Loading full solvated trajectory...")
u = mda.Universe(TPR_FILE, XTC_FILE)
protein = u.select_atoms('protein')

print(f"  Total atoms in system: {len(u.atoms)}")
print(f"  Protein atoms only:    {len(protein)}")
print(f"  Total frames:          {len(u.trajectory)}")

# Output paths
PROTEIN_XTC = f"{FOLDERS['protein_traj']}/protein_only.xtc"
PROTEIN_PDB = f"{FOLDERS['protein_traj']}/protein_only.pdb"

# Write protein only trajectory
print("Stripping water — please wait...")
with mda.Writer(PROTEIN_XTC, protein.n_atoms) as W:
    for ts in u.trajectory:
        W.write(protein)

protein.write(PROTEIN_PDB)

print(f"✅ Done!")
print(f"   Saved: protein_only.xtc")
print(f"   Saved: protein_only.pdb")
print(f"   Location: {FOLDERS['protein_traj']}")

Loading full solvated trajectory...
  Total atoms in system: 12295
  Protein atoms only:    166
  Total frames:          10001
Stripping water — please wait...
✅ Done!
   Saved: protein_only.xtc
   Saved: protein_only.pdb
   Location: /content/drive/MyDrive/RIN_Research/Chignolin/Run5/1_protein_trajectory


In [17]:
# ============================================================
# CELL 3 — EXTRACT FRAMES
# Input:  protein_only files from Cell 1
# Output: frame_XXXXX.pdb files → Drive frames folder
# ============================================================

u_frames = mda.Universe(PROTEIN_PDB, PROTEIN_XTC)
protein_atoms = u_frames.select_atoms('protein')

total_frames = len(u_frames.trajectory)
print(f"Extracting {total_frames} frames...")

count = 0
for ts in u_frames.trajectory:
    frame_number = ts.frame
    filename = os.path.join(
        FOLDERS['frames'],
        f'frame_{frame_number:06d}.pdb'
    )
    protein_atoms.write(filename)
    count += 1
    if count % 1000 == 0:
        print(f"  Progress: {count}/{total_frames} frames")

print(f"\n✅ Done! {count} frames saved")
print(f"   Location: {FOLDERS['frames']}")

Extracting 10001 frames...
  Progress: 1000/10001 frames
  Progress: 2000/10001 frames
  Progress: 3000/10001 frames
  Progress: 4000/10001 frames
  Progress: 5000/10001 frames
  Progress: 6000/10001 frames
  Progress: 7000/10001 frames
  Progress: 8000/10001 frames
  Progress: 9000/10001 frames
  Progress: 10000/10001 frames

✅ Done! 10001 frames saved
   Location: /content/drive/MyDrive/RIN_Research/Chignolin/Run5/2_pdb_frames


In [18]:
# ============================================================
# CELL 4 — BUILD RIN + FAST CENTRALITY
# Input:  pdb frames from Cell 3
# Output: fast_centrality_results.csv → Drive
# ============================================================

# Researcher's exact functions
def calculate_distance(residue_i, residue_j):
    try:
        atom_i = residue_i["CA"].coord
        atom_j = residue_j["CA"].coord
        return np.linalg.norm(atom_i - atom_j)
    except KeyError:
        return float('inf')

def construct_rin(pdb_file, cutoff=8.0):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)
    G = nx.Graph()
    for chain in structure.get_chains():
        for residue_i in chain:
            for residue_j in chain:
                if residue_i != residue_j:
                    distance = calculate_distance(residue_i, residue_j)
                    if distance < cutoff:
                        weight = 1.0 / (distance ** 2)
                        G.add_edge(
                            residue_i.get_id()[1],
                            residue_j.get_id()[1],
                            weight=weight
                        )
    return G

def compute_fast_centrality(G):
    results = {}
    results['degree'] = nx.degree_centrality(G)
    try:
        results['eigenvector'] = nx.eigenvector_centrality(
            G, max_iter=10000, tol=1e-3, weight='weight')
    except:
        results['eigenvector'] = nx.degree_centrality(G)
    try:
        results['closeness'] = nx.closeness_centrality(G)
    except:
        results['closeness'] = {n: 0 for n in G.nodes()}
    try:
        results['katz'] = nx.katz_centrality(
            G, alpha=0.1, max_iter=10000, tol=1e-3, weight='weight')
    except:
        results['katz'] = {n: 0 for n in G.nodes()}
    try:
        results['pagerank'] = nx.pagerank(
            G, alpha=0.85, weight='weight', max_iter=1000)
    except:
        results['pagerank'] = {n: 0 for n in G.nodes()}
    try:
        results['harmonic'] = nx.harmonic_centrality(G)
    except:
        results['harmonic'] = {n: 0 for n in G.nodes()}
    return results

# Get frames
pdb_files = sorted([
    f for f in os.listdir(FOLDERS['frames'])
    if f.endswith('.pdb')
])

print(f"Found {len(pdb_files)} frames")
print("Computing fast centrality measures...")

OUTPUT_CSV = f"{FOLDERS['centrality']}/fast_centrality_results.csv"

with open(OUTPUT_CSV, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([
        'frame', 'residue_id',
        'degree', 'eigenvector',
        'closeness', 'katz',
        'pagerank', 'harmonic',
        'num_edges'
    ])

    for idx, pdb_file in enumerate(pdb_files):
        frame_num = pdb_file.replace('frame_','').replace('.pdb','')
        full_path = os.path.join(FOLDERS['frames'], pdb_file)

        G = construct_rin(full_path, CUTOFF)
        if len(G.nodes()) == 0:
            continue

        # Save adjacency matrix
        residues = sorted(G.nodes())
        adj_matrix = nx.to_numpy_array(
            G, nodelist=residues, weight='weight')
        np.savetxt(
            f"{FOLDERS['matrices']}/adj_{frame_num}.txt",
            adj_matrix, fmt='%.4f'
        )

        centrality = compute_fast_centrality(G)

        for residue_id in residues:
            writer.writerow([
                frame_num, residue_id,
                round(centrality['degree'].get(residue_id, 0), 6),
                round(centrality['eigenvector'].get(residue_id, 0), 6),
                round(centrality['closeness'].get(residue_id, 0), 6),
                round(centrality['katz'].get(residue_id, 0), 6),
                round(centrality['pagerank'].get(residue_id, 0), 6),
                round(centrality['harmonic'].get(residue_id, 0), 6),
                G.number_of_edges()
            ])

        if idx % 500 == 0:
            print(f"  Progress: {idx}/{len(pdb_files)} "
                  f"({idx/len(pdb_files)*100:.1f}%)")

print(f"\n✅ Done!")
print(f"   Saved: fast_centrality_results.csv")
print(f"   Location: {FOLDERS['centrality']}")

Found 10001 frames
Computing fast centrality measures...
  Progress: 0/10001 (0.0%)
  Progress: 500/10001 (5.0%)
  Progress: 1000/10001 (10.0%)
  Progress: 1500/10001 (15.0%)
  Progress: 2000/10001 (20.0%)
  Progress: 2500/10001 (25.0%)
  Progress: 3000/10001 (30.0%)
  Progress: 3500/10001 (35.0%)
  Progress: 4000/10001 (40.0%)
  Progress: 4500/10001 (45.0%)
  Progress: 5000/10001 (50.0%)
  Progress: 5500/10001 (55.0%)
  Progress: 6000/10001 (60.0%)
  Progress: 6500/10001 (65.0%)
  Progress: 7000/10001 (70.0%)
  Progress: 7500/10001 (75.0%)
  Progress: 8000/10001 (80.0%)
  Progress: 8500/10001 (85.0%)
  Progress: 9000/10001 (90.0%)
  Progress: 9500/10001 (95.0%)
  Progress: 10000/10001 (100.0%)

✅ Done!
   Saved: fast_centrality_results.csv
   Location: /content/drive/MyDrive/RIN_Research/Chignolin/Run5/4_centrality


In [19]:
# ============================================================
# CELL 5 — SLOW CENTRALITY (BETWEENNESS + LOAD)
# Input:  pdb frames from Cell 3
# Output: slow_centrality_results.csv → Drive
# ============================================================

pdb_files = sorted([
    f for f in os.listdir(FOLDERS['frames'])
    if f.endswith('.pdb')
])

print(f"Found {len(pdb_files)} frames")
print("Computing betweenness + load centrality...")

OUTPUT_CSV = f"{FOLDERS['centrality']}/slow_centrality_results.csv"

with open(OUTPUT_CSV, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([
        'frame', 'residue_id',
        'betweenness', 'load',
        'num_edges'
    ])

for idx, pdb_file in enumerate(pdb_files):
    frame_num = pdb_file.replace('frame_','').replace('.pdb','')
    full_path = os.path.join(FOLDERS['frames'], pdb_file)

    try:
        G = construct_rin(full_path, CUTOFF)
        if len(G.nodes()) == 0:
            continue

        betweenness = nx.betweenness_centrality(
            G, weight='weight', normalized=True)
        load = nx.load_centrality(
            G, weight='weight', normalized=True)

        with open(OUTPUT_CSV, 'a', newline='') as csvfile:
            writer = csv.writer(csvfile)
            for residue_id in sorted(G.nodes()):
                writer.writerow([
                    frame_num, residue_id,
                    round(betweenness.get(residue_id, 0), 6),
                    round(load.get(residue_id, 0), 6),
                    G.number_of_edges()
                ])

        if idx % 500 == 0:
            print(f"  Progress: {idx}/{len(pdb_files)} "
                  f"({idx/len(pdb_files)*100:.1f}%)")

    except Exception as e:
        print(f"Error frame {frame_num}: {e}")
        continue

print(f"\n✅ Done!")
print(f"   Saved: slow_centrality_results.csv")
print(f"   Location: {FOLDERS['centrality']}")

Found 10001 frames
Computing betweenness + load centrality...
  Progress: 0/10001 (0.0%)
  Progress: 500/10001 (5.0%)
  Progress: 1000/10001 (10.0%)
  Progress: 1500/10001 (15.0%)
  Progress: 2000/10001 (20.0%)
  Progress: 2500/10001 (25.0%)
  Progress: 3000/10001 (30.0%)
  Progress: 3500/10001 (35.0%)
  Progress: 4000/10001 (40.0%)
  Progress: 4500/10001 (45.0%)
  Progress: 5000/10001 (50.0%)
  Progress: 5500/10001 (55.0%)
  Progress: 6000/10001 (60.0%)
  Progress: 6500/10001 (65.0%)
  Progress: 7000/10001 (70.0%)
  Progress: 7500/10001 (75.0%)
  Progress: 8000/10001 (80.0%)
  Progress: 8500/10001 (85.0%)
  Progress: 9000/10001 (90.0%)
  Progress: 9500/10001 (95.0%)
  Progress: 10000/10001 (100.0%)

✅ Done!
   Saved: slow_centrality_results.csv
   Location: /content/drive/MyDrive/RIN_Research/Chignolin/Run5/4_centrality


In [20]:
# ============================================================
# CELL 6 — MERGE CENTRALITY RESULTS
# Input:  fast + slow CSV from Cells 4 and 5
# Output: all_centrality_results.csv → Drive
# ============================================================

fast = pd.read_csv(
    f"{FOLDERS['centrality']}/fast_centrality_results.csv")
slow = pd.read_csv(
    f"{FOLDERS['centrality']}/slow_centrality_results.csv")

# Remove duplicates from slow
slow = slow.drop_duplicates(
    subset=['frame', 'residue_id'], keep='first')

# Merge on frame + residue_id
merged = fast.merge(
    slow[['frame', 'residue_id', 'betweenness', 'load']],
    on=['frame', 'residue_id'],
    how='left'
)

OUTPUT_CSV = f"{FOLDERS['centrality']}/all_centrality_results.csv"
merged.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Merged centrality results")
print(f"   Rows: {len(merged)}")
print(f"   Columns: {list(merged.columns)}")
print(f"   Saved: all_centrality_results.csv")
print(f"   Location: {FOLDERS['centrality']}")

✅ Merged centrality results
   Rows: 97542
   Columns: ['frame', 'residue_id', 'degree', 'eigenvector', 'closeness', 'katz', 'pagerank', 'harmonic', 'num_edges', 'betweenness', 'load']
   Saved: all_centrality_results.csv
   Location: /content/drive/MyDrive/RIN_Research/Chignolin/Run5/4_centrality


In [21]:
import warnings
warnings.filterwarnings('ignore')
# ============================================================
# CELL 7 — ORDER PARAMETERS
# Input:  TPR + XTC + native PDB
# Output: order_parameters.csv → Drive
# ============================================================

print("Computing order parameters...")

# ============================================================
# STEP 2A — RMSD with PBC fix + rotation alignment
# ============================================================

print("\n[1/3] Computing RMSD with PBC correction + alignment...")

# Load the FULL solvated trajectory (needed for PBC correction)
u_full = mda.Universe(
    TPR_FILE,
    XTC_FILE
)

# Load reference native structure
ref = mda.Universe(NATIVE_PDB)

# Select protein atoms from the full solvated system
protein_sel = u_full.select_atoms('protein')

# Select Cα atoms from reference
ref_ca = ref.select_atoms('name CA')

print(f"  Protein atoms in trajectory: {len(protein_sel)}")
print(f"  Cα atoms in reference: {len(ref_ca)}")

# Step 1 — Unwrap: fix PBC artifacts
unwrap_transform = unwrap(protein_sel)

# Step 2 — Fit: rotation + translation alignment
fit_transform = fit_rot_trans(
    protein_sel.select_atoms('name CA'),  # Use Cα atoms from mobile for alignment
    ref_ca                                # Use Cα atoms from reference for alignment
)

# Apply both transformations to the trajectory
u_full.trajectory.add_transformations(
    unwrap_transform,
    fit_transform
)

# Now compute RMSD — superposition=False because
# we already aligned with fit_rot_trans above
R = rms.RMSD(
    u_full,              # mobile trajectory
    ref,                 # reference structure
    select='name CA',    # measure RMSD on Cα atoms
    ref_select='name CA',
    superposition=False  # already aligned above
)

print("  Running RMSD calculation across all frames...")
R.run()

# Extract results
rmsd_values = R.results.rmsd[:, 2].tolist()
frames = R.results.rmsd[:, 0].astype(int).tolist()

rmsd_df = pd.DataFrame({
    'frame': frames,
    'rmsd': rmsd_values
})

print(f"  Min RMSD: {min(rmsd_values):.3f} Å")
print(f"  Max RMSD: {max(rmsd_values):.3f} Å")
print(f"  Avg RMSD: {np.mean(rmsd_values):.3f} Å")

# ============================================================
# STEP 2B — Radius of Gyration (Rg)
# ============================================================

print("\n[2/3] Computing Radius of Gyration...")

rg_values = []

# Use the already existing and transformed u_full universe for consistency
protein_atoms_rg = u_full.select_atoms('protein')

for ts in u_full.trajectory:
    coords = protein_atoms_rg.positions
    center = coords.mean(axis=0)
    rg = np.sqrt(((coords - center) ** 2).sum(axis=1).mean())
    rg_values.append(rg)

rg_df = pd.DataFrame({
    'frame': frames,
    'rg': rg_values
})

print(f"  Min Rg: {min(rg_values):.3f} Å")
print(f"  Max Rg: {max(rg_values):.3f} Å")
print(f"  Avg Rg: {np.mean(rg_values):.3f} Å")

# ============================================================
# STEP 2C — Fraction of Native Contacts (Q)
# ============================================================

print("\n[3/3] Computing Fraction of Native Contacts Q...")

def get_ca_coords(universe):
    ca = universe.select_atoms('name CA')
    return ca.positions.copy()

def get_native_contacts(native_pdb_path, cutoff=8.0):
    ref_q_universe = mda.Universe(native_pdb_path)
    coords = get_ca_coords(ref_q_universe)
    n = len(coords)
    native_pairs = []
    for i in range(n):
        for j in range(i+2, n):  # skip sequential neighbors
            dist = np.linalg.norm(coords[i] - coords[j])
            if dist < cutoff:
                native_pairs.append((i, j))
    return native_pairs, coords

native_pairs, native_coords = get_native_contacts(
    NATIVE_PDB,
    CUTOFF
)

print(f"  Native contacts found: {len(native_pairs)}")

# Use the already existing and transformed u_full universe for consistency
protein_ca_q = u_full.select_atoms('name CA')

q_values = []

for ts in u_full.trajectory:
    coords = protein_ca_q.positions.copy()
    if len(coords) != len(native_coords):
        q_values.append(0)
        continue

    formed = 0
    for i, j in native_pairs:
        # Check to prevent IndexError if selection changes or is problematic
        if i < len(coords) and j < len(coords):
            dist = np.linalg.norm(coords[i] - coords[j])
            if dist < CUTOFF:
                formed += 1

    q = formed / len(native_pairs) if len(native_pairs) > 0 else 0
    q_values.append(q)

q_df = pd.DataFrame({
    'frame': frames,
    'q': q_values
})

print(f"  Min Q: {min(q_values):.3f}")
print(f"  Max Q: {max(q_values):.3f}")
print(f"  Avg Q: {np.mean(q_values):.3f}")

# ============================================================
# SAVE ORDER PARAMETERS
# ────────────────────────────────────────────────────────────

# Combine all order parameters
order_params = pd.DataFrame({
    'frame': rmsd_df['frame'],
    'rmsd': rmsd_df['rmsd'],
    'rg': rg_df['rg'],
    'q': q_df['q']
})

OUTPUT_CSV = f"{FOLDERS['order_params']}/order_parameters.csv"
order_params.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Done!")
print(f"   Saved: order_parameters.csv")
print(f"   Location: {FOLDERS['order_params']}")

Computing order parameters...

[1/3] Computing RMSD with PBC correction + alignment...
  Protein atoms in trajectory: 166
  Cα atoms in reference: 10
  Running RMSD calculation across all frames...
  Min RMSD: 1.118 Å
  Max RMSD: 8.429 Å
  Avg RMSD: 1.864 Å

[2/3] Computing Radius of Gyration...
  Min Rg: 5.679 Å
  Max Rg: 11.276 Å
  Avg Rg: 6.463 Å

[3/3] Computing Fraction of Native Contacts Q...
  Native contacts found: 20
  Min Q: 0.400
  Max Q: 1.000
  Avg Q: 0.858

✅ Done!
   Saved: order_parameters.csv
   Location: /content/drive/MyDrive/RIN_Research/Chignolin/Run5/5_order_parameters


In [22]:
# ============================================================
# CELL 8 — SMOOTH + HUB ANALYSIS + TEMPORAL METRICS
# Input:  all_centrality_results.csv from Cell  6
# Output: smoothed_centrality.csv + temporal_metrics.csv
# ============================================================

SMOOTH_WINDOW  = 100   # rolling average window
HUB_PERCENTILE = 10    # top X% = hub

df = pd.read_csv(
    f"{FOLDERS['centrality']}/all_centrality_results.csv")

measures = ['degree','eigenvector','closeness',
            'katz','pagerank','harmonic']

# ── Smooth ───────────────────────────────────────────────────
print("Smoothing centrality...")
smoothed_dfs = []

for residue in sorted(df['residue_id'].unique()):
    res_data = df[df['residue_id']==residue].sort_values('frame').copy()
    for measure in measures:
        res_data[f'{measure}_smooth'] = (
            res_data[measure]
            .rolling(window=SMOOTH_WINDOW, center=True, min_periods=1)
            .mean()
        )
    smoothed_dfs.append(res_data)

smoothed_df = pd.concat(smoothed_dfs).sort_values(['frame','residue_id'])
smoothed_df.to_csv(
    f"{FOLDERS['centrality']}/smoothed_centrality.csv", index=False)
print("  Saved: smoothed_centrality.csv")

# ── Hub analysis ─────────────────────────────────────────────
print("Computing hub metrics...")
hub_threshold = np.percentile(
    df['eigenvector'], 100 - HUB_PERCENTILE)

temporal_results = []
total_frames = df['frame'].nunique()

for residue in sorted(df['residue_id'].unique()):
    res_data = df[df['residue_id']==residue].sort_values('frame')
    eig = res_data['eigenvector'].values
    frames_arr = res_data['frame'].values

    mean_c    = np.mean(eig)
    variance  = np.var(eig)
    is_hub    = eig >= hub_threshold
    persistence = is_hub.sum() / len(is_hub)

    onset_frame = 'Never permanent'
    for i in range(len(is_hub)):
        if is_hub[i:].all():
            onset_frame = int(frames_arr[i])
            break

    temporal_results.append({
        'residue_id'      : int(residue),
        'mean_centrality' : round(mean_c, 6),
        'variance'        : round(variance, 6),
        'persistence'     : round(persistence, 4),
        'onset_frame'     : onset_frame
    })

temporal_df = pd.DataFrame(temporal_results)
temporal_df.to_csv(
    f"{FOLDERS['centrality']}/temporal_metrics.csv", index=False)

print("  Saved: temporal_metrics.csv")
print("\n=== TEMPORAL METRICS ===")
print(temporal_df.to_string(index=False))
print(f"\n✅ Done!")

Smoothing centrality...
  Saved: smoothed_centrality.csv
Computing hub metrics...
  Saved: temporal_metrics.csv

=== TEMPORAL METRICS ===
 residue_id  mean_centrality  variance  persistence     onset_frame
          1         0.230899  0.010761       0.0241 Never permanent
          2         0.352700  0.011100       0.1441 Never permanent
          3         0.361326  0.012042       0.1520 Never permanent
          4         0.282220  0.011893       0.0832 Never permanent
          5         0.256072  0.012424       0.0652 Never permanent
          6         0.309263  0.013139       0.0994 Never permanent
          7         0.368974  0.010143       0.1462 Never permanent
          8         0.354638  0.011450       0.1860 Never permanent
          9         0.266350  0.014348       0.0856 Never permanent
         10         0.149762  0.006904       0.0045 Never permanent

✅ Done!


In [23]:
# ============================================================
# CELL 9 — CORRELATIONS
# Input:  all_centrality_results.csv + order_parameters.csv
# Output: correlations.csv → Drive
# ============================================================

df = pd.read_csv(
    f"{FOLDERS['centrality']}/all_centrality_results.csv")
order = pd.read_csv(
    f"{FOLDERS['order_params']}/order_parameters.csv")

measures = ['degree','eigenvector','closeness',
            'katz','pagerank','harmonic']

merged = df.merge(order, on='frame', how='inner')

correlation_results = []

for residue in sorted(df['residue_id'].unique()):
    res_data = merged[merged['residue_id']==residue]
    for measure in measures:
        correlation_results.append({
            'residue_id'   : int(residue),
            'measure'      : measure,
            'corr_with_Q'  : round(res_data[measure].corr(res_data['q']), 4),
            'corr_with_RMSD': round(res_data[measure].corr(res_data['rmsd']), 4),
            'corr_with_Rg' : round(res_data[measure].corr(res_data['rg']), 4),
        })

corr_df = pd.DataFrame(correlation_results)
corr_df.to_csv(
    f"{FOLDERS['order_params']}/correlations.csv", index=False)

print("=== EIGENVECTOR CORRELATIONS ===")
eig = corr_df[corr_df['measure']=='eigenvector']
for _, row in eig.iterrows():
    print(f"  Residue {int(row['residue_id']):2d}: "
          f"Q={row['corr_with_Q']:+.3f}  "
          f"RMSD={row['corr_with_RMSD']:+.3f}  "
          f"Rg={row['corr_with_Rg']:+.3f}")

print(f"\n✅ Saved: correlations.csv")
print(f"   Location: {FOLDERS['order_params']}")

=== EIGENVECTOR CORRELATIONS ===
  Residue  1: Q=+0.057  RMSD=-0.080  Rg=-0.044
  Residue  2: Q=+0.260  RMSD=-0.235  Rg=-0.108
  Residue  3: Q=+0.118  RMSD=-0.095  Rg=-0.041
  Residue  4: Q=-0.103  RMSD=+0.116  Rg=+0.068
  Residue  5: Q=-0.136  RMSD=+0.145  Rg=+0.094
  Residue  6: Q=-0.136  RMSD=+0.111  Rg=+0.058
  Residue  7: Q=-0.023  RMSD=+0.016  Rg=-0.020
  Residue  8: Q=+0.093  RMSD=-0.089  Rg=-0.064
  Residue  9: Q=+0.069  RMSD=-0.081  Rg=-0.055
  Residue 10: Q=-0.167  RMSD=+0.137  Rg=+0.019

✅ Saved: correlations.csv
   Location: /content/drive/MyDrive/RIN_Research/Chignolin/Run5/5_order_parameters


In [24]:
# ============================================================
# CELL 10 — PLOTS
# Input:  all result CSVs
# Output: all plots → Drive plots folder
# ============================================================

df       = pd.read_csv(f"{FOLDERS['centrality']}/all_centrality_results.csv")
order    = pd.read_csv(f"{FOLDERS['order_params']}/order_parameters.csv")
smoothed = pd.read_csv(f"{FOLDERS['centrality']}/smoothed_centrality.csv")
temporal = pd.read_csv(f"{FOLDERS['centrality']}/temporal_metrics.csv")
corr_df  = pd.read_csv(f"{FOLDERS['order_params']}/correlations.csv")

# ── Plot 1 — Order Parameters ────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14,10), sharex=True)
axes[0].plot(order['frame'], order['rmsd'],
             color='steelblue', alpha=0.7, linewidth=0.5)
axes[0].set_ylabel('RMSD (Å)')
axes[0].set_title('RMSD over time')
axes[1].plot(order['frame'], order['q'],
             color='coral', alpha=0.7, linewidth=0.5)
axes[1].set_ylabel('Q (native contacts)')
axes[1].set_title('Fraction of Native Contacts Q over time')
axes[2].plot(order['frame'], order['rg'],
             color='green', alpha=0.7, linewidth=0.5)
axes[2].set_ylabel('Rg (Å)')
axes[2].set_xlabel('Frame')
axes[2].set_title('Radius of Gyration over time')
plt.tight_layout()
plt.savefig(f"{FOLDERS['plots']}/order_parameters.png",
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: order_parameters.png")

# ── Plot 2 — Smoothed Centrality ─────────────────────────────
plt.figure(figsize=(14,6))
for res in sorted(smoothed['residue_id'].unique()):
    res_data = smoothed[smoothed['residue_id']==res].sort_values('frame')
    plt.plot(res_data['frame'], res_data['eigenvector_smooth'],
             label=f'Res {int(res)}', alpha=0.8)
plt.xlabel('Frame')
plt.ylabel('Smoothed Eigenvector Centrality')
plt.title(f'Smoothed Centrality Over Time — {PROTEIN_NAME} {RUN_NAME}')
plt.legend(bbox_to_anchor=(1.05,1), loc='upper left')
plt.tight_layout()
plt.savefig(f"{FOLDERS['plots']}/smoothed_centrality.png",
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: smoothed_centrality.png")

# ── Plot 3 — Hub Persistence ─────────────────────────────────
plt.figure(figsize=(10,5))
persistence_vals = temporal['persistence'].tolist()
residue_labels   = [f'Res {int(r)}' for r in temporal['residue_id']]
colors = ['red' if p > 0.5 else 'steelblue' for p in persistence_vals]
plt.bar(residue_labels, persistence_vals, color=colors, alpha=0.8)
plt.xlabel('Residue')
plt.ylabel('Hub Persistence')
plt.title(f'Hub Persistence — {PROTEIN_NAME} {RUN_NAME}')
plt.tight_layout()
plt.savefig(f"{FOLDERS['plots']}/hub_persistence.png",
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: hub_persistence.png")

# ── Plot 4 — Correlation Heatmap ─────────────────────────────
eig_corr = corr_df[corr_df['measure']=='eigenvector'].set_index('residue_id')
heatmap_data = eig_corr[['corr_with_Q','corr_with_RMSD','corr_with_Rg']].values

plt.figure(figsize=(8,6))
plt.imshow(heatmap_data, cmap='RdBu', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(label='Pearson Correlation')
plt.xticks(range(3), ['Q','RMSD','Rg'])
plt.yticks(range(len(eig_corr)),
           [f'Res {int(r)}' for r in eig_corr.index])
plt.title(f'Eigenvector Centrality Correlation\nwith Folding Order Parameters')
plt.tight_layout()
plt.savefig(f"{FOLDERS['plots']}/correlation_heatmap.png",
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: correlation_heatmap.png")

# ── Plot 5 — Network Density ─────────────────────────────────
edges_over_time = df.groupby('frame')['num_edges'].first()
plt.figure(figsize=(14,4))
plt.plot(edges_over_time.index, edges_over_time.values,
         color='steelblue', alpha=0.7)
plt.xlabel('Frame')
plt.ylabel('Number of Edges')
plt.title(f'Network Density Over Time — {PROTEIN_NAME} {RUN_NAME}')
plt.tight_layout()
plt.savefig(f"{FOLDERS['plots']}/network_density.png",
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: network_density.png")

print(f"\n✅ All plots saved to: {FOLDERS['plots']}")

Saved: order_parameters.png
Saved: smoothed_centrality.png
Saved: hub_persistence.png
Saved: correlation_heatmap.png
Saved: network_density.png

✅ All plots saved to: /content/drive/MyDrive/RIN_Research/Chignolin/Run5/6_plots


In [25]:
# ============================================================
# CELL 11 — SUMMARY REPORT
# Input:  all result CSVs
# Output: summary.txt → Drive
# ============================================================

df       = pd.read_csv(f"{FOLDERS['centrality']}/all_centrality_results.csv")
order    = pd.read_csv(f"{FOLDERS['order_params']}/order_parameters.csv")
temporal = pd.read_csv(f"{FOLDERS['centrality']}/temporal_metrics.csv")
corr_df  = pd.read_csv(f"{FOLDERS['order_params']}/correlations.csv")

measures = ['degree','eigenvector','closeness',
            'katz','pagerank','harmonic']

avg = df.groupby('residue_id')[measures].mean()

lines = []
lines.append("=" * 60)
lines.append(f"ANALYSIS SUMMARY")
lines.append(f"Protein: {PROTEIN_NAME} | Run: {RUN_NAME}")
lines.append("=" * 60)

lines.append("\nTOP RESIDUE PER CENTRALITY MEASURE:")
for measure in measures:
    top = avg[measure].idxmax()
    score = avg[measure].max()
    top_per_frame = df.loc[df.groupby('frame')[measure].idxmax()]
    counts = top_per_frame['residue_id'].value_counts()
    pct = counts.max() / df['frame'].nunique() * 100
    lines.append(
        f"  {measure:12}: Residue {int(top)} "
        f"(avg={score:.4f}, dominant in {pct:.1f}% of frames)")

lines.append("\nORDER PARAMETERS:")
lines.append(f"  RMSD — Min:{order['rmsd'].min():.3f} "
             f"Max:{order['rmsd'].max():.3f} "
             f"Avg:{order['rmsd'].mean():.3f} Å")
lines.append(f"  Q    — Min:{order['q'].min():.3f} "
             f"Max:{order['q'].max():.3f} "
             f"Avg:{order['q'].mean():.3f}")
lines.append(f"  Rg   — Min:{order['rg'].min():.3f} "
             f"Max:{order['rg'].max():.3f} "
             f"Avg:{order['rg'].mean():.3f} Å")

lines.append("\nMOST PERSISTENT HUB:")
top_hub = temporal.loc[temporal['persistence'].idxmax()]
lines.append(
    f"  Residue {int(top_hub['residue_id'])} — "
    f"hub in {top_hub['persistence']*100:.1f}% of frames")

lines.append("\nEIGENVECTOR CENTRALITY CORRELATIONS:")
eig = corr_df[corr_df['measure']=='eigenvector']
for _, row in eig.iterrows():
    lines.append(
        f"  Residue {int(row['residue_id']):2d}: "
        f"Q={row['corr_with_Q']:+.3f}  "
        f"RMSD={row['corr_with_RMSD']:+.3f}  "
        f"Rg={row['corr_with_Rg']:+.3f}")

lines.append("\nNETWORK DENSITY:")
edges = df.groupby('frame')['num_edges'].first()
lines.append(f"  Min edges: {edges.min()}")
lines.append(f"  Max edges: {edges.max()}")
lines.append(f"  Avg edges: {edges.mean():.1f}")
lines.append("=" * 60)

summary_text = "\n".join(lines)
print(summary_text)

OUTPUT_TXT = f"{FOLDERS['summary']}/summary_{PROTEIN_NAME}_{RUN_NAME}.txt"
with open(OUTPUT_TXT, 'w') as f:
    f.write(summary_text)

print(f"\n✅ Summary saved: {OUTPUT_TXT}")

ANALYSIS SUMMARY
Protein: Chignolin | Run: Run5

TOP RESIDUE PER CENTRALITY MEASURE:
  degree      : Residue 3 (avg=0.6928, dominant in 45.7% of frames)
  eigenvector : Residue 7 (avg=0.3690, dominant in 34.1% of frames)
  closeness   : Residue 3 (avg=0.7134, dominant in 45.5% of frames)
  katz        : Residue 3 (avg=0.3218, dominant in 40.3% of frames)
  pagerank    : Residue 3 (avg=0.1182, dominant in 35.7% of frames)
  harmonic    : Residue 3 (avg=6.6019, dominant in 45.5% of frames)

ORDER PARAMETERS:
  RMSD — Min:1.118 Max:8.429 Avg:1.864 Å
  Q    — Min:0.400 Max:1.000 Avg:0.858
  Rg   — Min:5.679 Max:11.276 Avg:6.463 Å

MOST PERSISTENT HUB:
  Residue 8 — hub in 18.6% of frames

EIGENVECTOR CENTRALITY CORRELATIONS:
  Residue  1: Q=+0.057  RMSD=-0.080  Rg=-0.044
  Residue  2: Q=+0.260  RMSD=-0.235  Rg=-0.108
  Residue  3: Q=+0.118  RMSD=-0.095  Rg=-0.041
  Residue  4: Q=-0.103  RMSD=+0.116  Rg=+0.068
  Residue  5: Q=-0.136  RMSD=+0.145  Rg=+0.094
  Residue  6: Q=-0.136  RMSD=+0.11

In [26]:
# ============================================================
# CELL 12 — VALIDATION
# Automatically checks if results are biologically sensible
# ============================================================

import scipy.stats as stats

df    = pd.read_csv(f"{FOLDERS['centrality']}/all_centrality_results.csv")
order = pd.read_csv(f"{FOLDERS['order_params']}/order_parameters.csv")
corr  = pd.read_csv(f"{FOLDERS['order_params']}/correlations.csv")

passed = 0
failed = 0
warnings_list = []

print("=" * 60)
print("VALIDATION CHECKS")
print(f"Protein: {PROTEIN_NAME} | Run: {RUN_NAME}")
print("=" * 60)

# ── Check 1 — RMSD and Q are negatively correlated ───────────
rmsd_q_corr = order['rmsd'].corr(order['q'])
if rmsd_q_corr < -0.3:
    print(f"✅ Check 1 PASSED: RMSD vs Q correlation = {rmsd_q_corr:.3f}")
    print(f"   (negative as expected — when Q rises RMSD drops)")
    passed += 1
else:
    print(f"❌ Check 1 FAILED: RMSD vs Q correlation = {rmsd_q_corr:.3f}")
    print(f"   (expected < -0.3 — RMSD and Q should move opposite)")
    failed += 1

# ── Check 2 — Rg and Q are negatively correlated ─────────────
rg_q_corr = order['rg'].corr(order['q'])
if rg_q_corr < -0.3:
    print(f"\n✅ Check 2 PASSED: Rg vs Q correlation = {rg_q_corr:.3f}")
    print(f"   (negative as expected — when Q rises Rg drops)")
    passed += 1
else:
    print(f"\n❌ Check 2 FAILED: Rg vs Q correlation = {rg_q_corr:.3f}")
    print(f"   (expected < -0.3)")
    failed += 1

# ── Check 3 — RMSD folded state is physically reasonable ─────
min_rmsd = order['rmsd'].min()
if min_rmsd < 3.0:
    print(f"\n✅ Check 3 PASSED: Min RMSD = {min_rmsd:.3f} Å")
    print(f"   (below 3Å — protein reached folded state)")
    passed += 1
else:
    print(f"\n❌ Check 3 FAILED: Min RMSD = {min_rmsd:.3f} Å")
    print(f"   (expected below 3Å — protein may not have folded)")
    failed += 1

# ── Check 4 — Q reaches near 1.0 at some point ───────────────
max_q = order['q'].max()
if max_q > 0.8:
    print(f"\n✅ Check 4 PASSED: Max Q = {max_q:.3f}")
    print(f"   (above 0.8 — protein achieved near-native contacts)")
    passed += 1
else:
    print(f"\n❌ Check 4 FAILED: Max Q = {max_q:.3f}")
    print(f"   (expected above 0.8 — protein may not have folded)")
    failed += 1

# ── Check 5 — Residue 3 is top eigenvector hub ───────────────
avg_eig = df.groupby('residue_id')['eigenvector'].mean()
top_residue = avg_eig.idxmax()
if top_residue == 3:
    print(f"\n✅ Check 5 PASSED: Top eigenvector hub = Residue {int(top_residue)}")
    print(f"   (ASP3 — matches published Chignolin literature)")
    passed += 1
else:
    print(f"\n⚠️  Check 5 WARNING: Top eigenvector hub = Residue {int(top_residue)}")
    print(f"   (expected Residue 3/ASP3 — different result worth investigating)")
    warnings_list.append(f"Top hub is Residue {int(top_residue)} not Residue 3")

# ── Check 6 — Terminal residues rank low ─────────────────────
res1_rank  = avg_eig.rank(ascending=False)[1]
res10_rank = avg_eig.rank(ascending=False)[10]
if res1_rank >= 8 and res10_rank >= 8:
    print(f"\n✅ Check 6 PASSED: Terminal residues rank low")
    print(f"   (Res 1 rank={int(res1_rank)}, "
          f"Res 10 rank={int(res10_rank)} out of {N_RESIDUES})")
    passed += 1
else:
    print(f"\n⚠️  Check 6 WARNING: Terminal residues rank unexpectedly high")
    print(f"   (Res 1 rank={int(res1_rank)}, "
          f"Res 10 rank={int(res10_rank)})")
    warnings_list.append("Terminal residues ranking unexpectedly high")

# ── Check 7 — Residue 3 positively correlates with Q ─────────
res3_q = corr[
    (corr['residue_id']==3) &
    (corr['measure']=='eigenvector')
]['corr_with_Q'].values[0]

if res3_q > 0.1:
    print(f"\n✅ Check 7 PASSED: Residue 3 vs Q correlation = {res3_q:+.3f}")
    print(f"   (positive — centrality rises as protein folds)")
    passed += 1
else:
    print(f"\n⚠️  Check 7 WARNING: Residue 3 vs Q = {res3_q:+.3f}")
    print(f"   (expected positive correlation with Q)")
    warnings_list.append(f"Residue 3 Q correlation weak: {res3_q:.3f}")

# ── Check 8 — Network density makes sense ────────────────────
min_edges = df.groupby('frame')['num_edges'].first().min()
max_edges = df.groupby('frame')['num_edges'].first().max()
max_possible = N_RESIDUES * (N_RESIDUES - 1) // 2

if min_edges >= 1 and max_edges <= max_possible:
    print(f"\n✅ Check 8 PASSED: Edge range {min_edges}-{max_edges}")
    print(f"   (within valid range 1-{max_possible} for "
          f"{N_RESIDUES} residues)")
    passed += 1
else:
    print(f"\n❌ Check 8 FAILED: Edge range {min_edges}-{max_edges}")
    print(f"   (outside valid range for {N_RESIDUES} residues)")
    failed += 1

# ── Check 9 — Lindorff-Larsen RMSD comparison ────────────────
# For Chignolin: folded state RMSD should be ~1.0 Å
# From Lindorff-Larsen 2011 Figure 1: cln025 = 1.0 Å
folded_frames = order[order['q'] > 0.8]
if len(folded_frames) > 0:
    folded_rmsd = folded_frames['rmsd'].mean()
    if folded_rmsd < 3.0:
        print(f"\n✅ Check 9 PASSED: Avg RMSD in folded state "
              f"(Q>0.8) = {folded_rmsd:.3f} Å")
        print(f"   (consistent with Lindorff-Larsen 2011: ~1.0 Å for CLN025)")
        passed += 1
    else:
        print(f"\n⚠️  Check 9 WARNING: Avg RMSD in folded state = "
              f"{folded_rmsd:.3f} Å")
        print(f"   (higher than expected ~1-2 Å)")
        warnings_list.append(
            f"Folded state RMSD {folded_rmsd:.3f} Å higher than expected")
else:
    print(f"\n⚠️  Check 9 SKIPPED: No frames found with Q > 0.8")
    warnings_list.append("No clearly folded frames found (Q > 0.8)")

# ── Check 10 — Spot check one random frame ───────────────────
import random
random_frame = random.choice(
    sorted(df['frame'].unique()))
frame_data = df[df['frame']==random_frame]
order_data = order[order['frame']==random_frame]

print(f"\n✅ Check 10 — Random frame spot check")
print(f"   Frame: {random_frame}")
if len(order_data) > 0:
    print(f"   RMSD:  {order_data['rmsd'].values[0]:.3f} Å")
    print(f"   Q:     {order_data['q'].values[0]:.3f}")
    print(f"   Rg:    {order_data['rg'].values[0]:.3f} Å")
print(f"   Edges: {frame_data['num_edges'].values[0]}")
print(f"   Top residue (eigenvector): "
      f"Residue {int(frame_data.loc[frame_data['eigenvector'].idxmax(), 'residue_id'])}")
print(f"   → Manually check frame_{random_frame:06d}.pdb in ChimeraX")
print(f"     to visually confirm these values make sense")

# ── Final Summary ─────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"VALIDATION SUMMARY: {passed} passed / "
      f"{failed} failed / {len(warnings_list)} warnings")
print("=" * 60)

if failed == 0 and len(warnings_list) == 0:
    print("🎉 ALL CHECKS PASSED — Results look biologically valid")
elif failed == 0:
    print("⚠️  PASSED WITH WARNINGS — Review warnings above")
    for w in warnings_list:
        print(f"   • {w}")
else:
    print("❌ SOME CHECKS FAILED — Review errors above")
    for w in warnings_list:
        print(f"   • {w}")

# Save validation report
report_lines = [
    f"VALIDATION REPORT",
    f"Protein: {PROTEIN_NAME} | Run: {RUN_NAME}",
    f"Checks passed: {passed}",
    f"Checks failed: {failed}",
    f"Warnings: {len(warnings_list)}",
]
for w in warnings_list:
    report_lines.append(f"Warning: {w}")

with open(f"{FOLDERS['summary']}/validation_report.txt", 'w') as f:
    f.write('\n'.join(report_lines))

print(f"\n✅ Validation report saved to Drive")

VALIDATION CHECKS
Protein: Chignolin | Run: Run5
✅ Check 1 PASSED: RMSD vs Q correlation = -0.873
   (negative as expected — when Q rises RMSD drops)

✅ Check 2 PASSED: Rg vs Q correlation = -0.557
   (negative as expected — when Q rises Rg drops)

✅ Check 3 PASSED: Min RMSD = 1.118 Å
   (below 3Å — protein reached folded state)

✅ Check 4 PASSED: Max Q = 1.000
   (above 0.8 — protein achieved near-native contacts)

⚠️  Check 5 WARNING: Top eigenvector hub = Residue 7
   (expected Residue 3/ASP3 — different result worth investigating)

✅ Check 6 PASSED: Terminal residues rank low
   (Res 1 rank=9, Res 10 rank=10 out of 10)

✅ Check 7 PASSED: Residue 3 vs Q correlation = +0.118
   (positive — centrality rises as protein folds)

✅ Check 8 PASSED: Edge range 4-35
   (within valid range 1-45 for 10 residues)

✅ Check 9 PASSED: Avg RMSD in folded state (Q>0.8) = 1.730 Å
   (consistent with Lindorff-Larsen 2011: ~1.0 Å for CLN025)

✅ Check 10 — Random frame spot check
   Frame: 5382
   RMSD:

In [27]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/RIN_Research/Chignolin/Run5/4_centrality/all_centrality_results.csv')

print("=== RUN 5 AVERAGE CENTRALITY ===")
measures = ['degree', 'eigenvector', 'closeness',
            'katz', 'pagerank', 'harmonic']

avg = df.groupby('residue_id')[measures].mean()
print(avg.round(4))

print("\n=== TOP RESIDUE PER MEASURE ===")
for measure in measures:
    top = avg[measure].idxmax()
    score = avg[measure].max()
    print(f"{measure:12}: Residue {int(top)} — {score:.4f}")

print("\n=== RESIDUE 3 vs RESIDUE 7 EIGENVECTOR ===")
res3 = avg.loc[3, 'eigenvector']
res7 = avg.loc[7, 'eigenvector']
diff = abs(res3 - res7)
print(f"Residue 3: {res3:.6f}")
print(f"Residue 7: {res7:.6f}")
print(f"Difference: {diff:.6f}")

=== RUN 5 AVERAGE CENTRALITY ===
            degree  eigenvector  closeness    katz  pagerank  harmonic
residue_id                                                            
1           0.4737       0.2309     0.5696  0.3184    0.0874    5.5482
2           0.6314       0.3527     0.6713  0.3216    0.1161    6.3711
3           0.6928       0.3613     0.7134  0.3218    0.1182    6.6019
4           0.4891       0.2822     0.5539  0.3195    0.0955    5.5407
5           0.3948       0.2561     0.5036  0.3189    0.0913    5.0498
6           0.5807       0.3093     0.6339  0.3202    0.1032    6.0507
7           0.6657       0.3690     0.6940  0.3217    0.1161    6.5003
8           0.6527       0.3546     0.6862  0.3216    0.1169    6.4620
9           0.5367       0.2663     0.6035  0.3202    0.1075    5.7946
10          0.3075       0.1498     0.4425  0.3164    0.0701    4.5303

=== TOP RESIDUE PER MEASURE ===
degree      : Residue 3 — 0.6928
eigenvector : Residue 7 — 0.3690
closeness   : Re